In [0]:
CREATE OR REPLACE TABLE ouro_passagens AS
WITH base_calculada AS (
    SELECT 
        p.data_partida,
        date_format(p.data_hora_partida, 'HH:mm') AS horario_saida,
        hour(p.data_hora_partida) AS hora_num, 
        trim(p.empresa) AS empresa,
        p.origem,
        p.destino,
        p.classe_servico,
        p.assentos_disponiveis,
        CASE 
            WHEN p.classe_servico LIKE '%Convencional%' THEN 46
            WHEN p.classe_servico LIKE '%Executivo%' THEN 42
            WHEN p.classe_servico LIKE '%Semi%' THEN 42
            WHEN p.classe_servico LIKE '%Leito%' THEN 28
            WHEN p.classe_servico LIKE '%Cama%' THEN 12
            ELSE 40
        END AS capacidade_maxima,
        p.preco_passagem,
        p.data_processamento,
        ROW_NUMBER() OVER (
            PARTITION BY p.origem, p.destino, p.data_hora_partida, p.empresa, p.classe_servico 
            ORDER BY p.data_processamento DESC
        ) as rn_final
    FROM prata_passagens p
),
metricas AS (
    SELECT 
        *,
        GREATEST(COALESCE(capacidade_maxima - assentos_disponiveis, 0), 0) AS assentos_ocupados,
        CASE 
            WHEN hora_num BETWEEN 0 AND 5  THEN 'Madrugada'
            WHEN hora_num BETWEEN 6 AND 11 THEN 'Manhã'
            WHEN hora_num BETWEEN 12 AND 17 THEN 'Tarde'
            ELSE 'Noite'
        END AS turno,
        CASE 
            WHEN hora_num BETWEEN 0 AND 5  THEN 1
            WHEN hora_num BETWEEN 6 AND 11 THEN 2
            WHEN hora_num BETWEEN 12 AND 17 THEN 3
            ELSE 4
        END AS turno_id
    FROM base_calculada
    WHERE rn_final = 1
)
SELECT 
    data_partida,
    horario_saida,
    hora_num,
    empresa,
    origem,
    destino,
    classe_servico,
    assentos_disponiveis,
    capacidade_maxima,
    preco_passagem,
    data_processamento,
    assentos_ocupados,
    ROUND(
        LEAST(CAST(assentos_ocupados AS DOUBLE) / NULLIF(capacidade_maxima, 0), 1.0),
        4
    ) AS taxa_ocupacao,
    turno,
    turno_id
FROM metricas;

SELECT * FROM ouro_passagens LIMIT 10;

data_partida,horario_saida,hora_num,empresa,origem,destino,classe_servico,assentos_disponiveis,capacidade_maxima,preco_passagem,data_processamento,assentos_ocupados,taxa_ocupacao,turno,turno_id
2026-04-26,13:20,13,Util,Belo Horizonte,Rio de Janeiro,SemiLeito,7,42,109.99,2026-04-26T11:51:04.656Z,35,0.8333,Tarde,3
2026-04-26,14:00,14,Cometa,Belo Horizonte,Rio de Janeiro,Convencional,29,46,219.99,2026-04-26T11:51:04.656Z,17,0.3696,Tarde,3
2026-04-26,16:00,16,Cometa,Belo Horizonte,Rio de Janeiro,SemiLeito,6,42,229.99,2026-04-26T11:51:04.656Z,36,0.8571,Tarde,3
2026-04-26,16:30,16,Util,Belo Horizonte,Rio de Janeiro,SemiLeito,18,42,219.99,2026-04-26T11:51:04.656Z,24,0.5714,Tarde,3
2026-04-26,18:00,18,Util,Belo Horizonte,Rio de Janeiro,Executivo,2,42,219.99,2026-04-26T11:51:04.656Z,40,0.9524,Noite,4
2026-04-26,18:15,18,Util,Belo Horizonte,Rio de Janeiro,Executivo,24,42,219.99,2026-04-26T11:51:04.656Z,18,0.4286,Noite,4
2026-04-26,18:30,18,Auto Viação São Luiz,Belo Horizonte,Rio de Janeiro,Convencional,36,46,226.55,2026-04-26T11:51:04.656Z,10,0.2174,Noite,4
2026-04-26,19:30,19,Auto Viação São Luiz,Belo Horizonte,Rio de Janeiro,Convencional,39,46,226.55,2026-04-26T11:51:04.656Z,7,0.1522,Noite,4
2026-04-26,19:45,19,Auto Viação São Luiz,Belo Horizonte,Rio de Janeiro,Convencional,41,46,226.55,2026-04-26T11:51:04.656Z,5,0.1087,Noite,4
2026-04-26,20:45,20,Auto Viação São Luiz,Belo Horizonte,Rio de Janeiro,Convencional,34,46,226.55,2026-04-26T11:51:04.656Z,12,0.2609,Noite,4
